# ADS Bibliometric Analysis Pipeline

This notebook is the single entry point for the `ads_bib` package.
It executes all steps sequentially, with configuration cells between steps
so that downstream parameters can be set based on upstream results.

**Pipeline Steps:**
1. Search & Export â€” Query ADS API, resolve bibcodes to metadata
2. Translation â€” Detect languages, translate non-English titles/abstracts
3. Tokenization â€” Create full_text, tokenize with spaCy
4. AND â€” Author Name Disambiguation (placeholder)
5. Topic Modeling & Curation â€” BERTopic + datamapplot + cluster removal
6. Citation Networks â€” Direct, Co-Citation, Bibliographic Coupling, Author Co-Citation

---
## Setup

In [1]:
import os
from pathlib import Path

# Initialize paths (relative to this notebook's location)
from ads_bib.config import init_paths, load_env
from ads_bib.run_manager import RunManager
from ads_bib._utils.costs import CostTracker

paths = init_paths()  # uses CWD = notebook directory
load_env()

# Cost tracker for all API calls (OpenRouter, etc.)
tracker = CostTracker()

# Verify
print(f"Project root: {paths['project_root']}")
print(f"Data dir:     {paths['data']}")

# If you see a spaCy version warning later, update the model:
# !python -m spacy download en_core_web_lg

# --- INITIALIZE RUN ---
# This tracks all parameters and saves outputs to a unique folder
run = RunManager(run_name='ADS_Curation_Run')


Project root: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline
Data dir:     c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\data
Run initialized: run_20260223_133015_ADS_Curation_Run
All run outputs will be saved to: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260223_133015_ADS_Curation_Run


---
## Global Run Control

Set run-level switches here. Phase-specific parameters are configured in each phase section below.


In [2]:
import random
import numpy as np

# --- RUN CONTROL ---
# 0 = Run everything (Search to End)
# 4 = Load processed data (Skip Search, Translate, Tokenize) -> Start at Topic Modeling
START_AT_PHASE = 5
FORCE_REFRESH = False

# Deterministic seed for notebook-side randomness (sampling + reduction params)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Shared OpenRouter pricing mode (used in translation, embeddings, topic labeling)
OPENROUTER_COST_MODE = "hybrid"  # 'hybrid', 'strict', 'fast'


---
# Phase 1: Search & Export

Query the NASA ADS API for publications matching a search string,
then resolve all bibcodes (publications + references) into full metadata.

### 1.1 Search Configuration

Set query, years, and retrieval limits for your research question.
These values define the corpus scope for all downstream steps.


In [ ]:
# # --- SEARCH CONFIGURATION ---
# # Modify the query string for your research question.
# # See: https://ui.adsabs.harvard.edu/help/search/search-syntax

# ADS_TOKEN = os.getenv("ADS_API_KEY")

# Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"
# Set_B = f"citations({Set_A}) AND year:1915-2000"
# Set_C = f"citations(citations({Set_A})) AND year:1915-2000"
# Set_D = f"({Set_A}) OR ({Set_B}) OR ({Set_C})"
# Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

# gravity_relativity_terms = [
#     '(general AND relativi*)', 'gravit*', '(allgemein* AND relativi*)',
#     '"relativité générale"', '"teoria della relatività"',
#     '"Gravité quantique"', '"Gravità quantistica"',
#     '(einheitlich* AND feld*)', 'Quantengravitation', '"champ unifié"',
#     '(unified AND field*)', '"quantum gravity"', 'cosmo*', 'Kosmo*',
# ]
# Set_E = f"abs:({' OR '.join(gravity_relativity_terms)}) AND year:1915-2000"

# # Example quick focus query
# # QUERY = 'author:"Treder, H*"'
# # Alternative broader query:
# QUERY = f"({Set_D}) OR ({Set_T}) OR ({Set_E})"


In [ ]:
# ...existing code...
# --- SEARCH CONFIGURATION ---
# Modify the query string for your research question.
# See: https://ui.adsabs.harvard.edu/help/search/search-syntax

ADS_TOKEN = os.getenv("ADS_API_KEY")

# 1. Historischer Seed: Einsteins Publikationen (1911-1955)
Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"

# 2. Direkte Rezeption: 1-Hop Zitationen ab 1915
Set_B = f"citations({Set_A}) AND year:1915-2000"

# 3. Strukturelle Anker-Begriffe (Mehrsprachig: EN, DE, FR, IT)
generic_anchors = [
    'relativ*', 'einstein*', 
    'spacetime', '"space-time"', 'raumzeit', '"espace-temps"', 'spaziotempo', '"spazio-tempo"',
    'tensor*', 'tenseur*', 
    'metric*', 'metrik*', 'metriqu*', 
    'curvature', 'krümmung', 'kruemmung', 'courbure', 'curvatura',
    'geodesic*', 'geodät*', 'geodaet*', 'geodesiqu*', 'geodetic*'
]

# 4. Indirekte Rezeption: 2-Hop Zitationen ab 1915 (Gefiltert durch Anker)
Set_C = f"citations(citations({Set_A})) AND year:1915-2000 AND abs:({' OR '.join(generic_anchors)})"

# 5. Treder Library
Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

# 6. Text-Suche: Core-Begriffe und verankerte breite Begriffe
grg_core_terms = [
    '(general AND relativi*)', '(allgemein* AND relativi*)',
    '"relativité générale"', '"relatività generale"',
    '"Gravité quantique"', '"Gravità quantistica"', 'Quantengravitation', '"quantum gravity"',
    '(einheitlich* AND feld*)', '"champ unifié"', '(unified AND field*)'
]

broad_terms = ['gravit*', 'cosmo*', 'Kosmo*']
anchored_broad = f"({' OR '.join(broad_terms)}) AND ({' OR '.join(generic_anchors)})"

Set_E = f"abs:({' OR '.join(grg_core_terms)} OR ({anchored_broad})) AND year:1911-2000"

# Finale Query
QUERY = f"({Set_A}) OR ({Set_B}) OR ({Set_C}) OR ({Set_T}) OR ({Set_E})"
# ...existing code...

### 1.2 Execute Search

Run the ADS search and persist bibcodes/results to this run folder.
This creates a reproducible input snapshot for later phases.


In [ ]:
if START_AT_PHASE <= 1:
    from ads_bib.search import search_ads, save_search_results
    from ads_bib._utils.io import load_pickle
    
    latest = paths["raw"] / "search_results_latest.pkl"
    
    if FORCE_REFRESH or not latest.exists():
        bibcodes, references, esources, fulltext_urls = search_ads(QUERY, ADS_TOKEN)
        save_search_results((bibcodes, references, esources, fulltext_urls), paths["raw"])
    else:
        print(f"Loading cached results: {latest.name}")
        bibcodes, references, esources, fulltext_urls = load_pickle(latest)
    
    print(f"\nBibcodes:        {len(bibcodes):,}")
    print(f"Unique refs:     {len(set(r for rl in references for r in rl)):,}")
    print(f"Total refs:      {sum(len(rl) for rl in references):,}")
else:
    print(f"Skipping Phase 1 Search (START_AT_PHASE={START_AT_PHASE})")

### 1.3 Export & Resolve Metadata

Resolve publications and references to structured metadata tables.
This normalizes schemas before translation/tokenization/topic modeling.


In [ ]:
if START_AT_PHASE <= 1:
    from ads_bib.export import resolve_dataset
    from ads_bib._utils.io import save_json_lines, load_json_lines
    
    pubs_path = paths["raw"] / "publications.json"
    refs_path = paths["raw"] / "references.json"
    
    if FORCE_REFRESH or not pubs_path.exists():
        publications, refs = resolve_dataset(
            bibcodes, references, esources, fulltext_urls, ADS_TOKEN
        )
        save_json_lines(publications, pubs_path)
        save_json_lines(refs, refs_path)
    else:
        print("Loading cached exports ...")
        publications = load_json_lines(pubs_path)
        refs = load_json_lines(refs_path)
    
    print(f"\nPublications: {len(publications):,}")
    print(f"References:   {len(refs):,}")
    publications.info()

In [ ]:
if START_AT_PHASE <= 1:
    display(publications.head(50))

---
# Phase 2: Translation

Detect non-English titles and abstracts with fasttext,
then translate them using either OpenRouter (API) or a local HuggingFace model.

### 2.1 Translation Configuration

Choose provider/model and translation target language.
Keep settings explicit so costs and outputs are reproducible.


In [ ]:
# --- TRANSLATION CONFIGURATION ---
# Provider options:
# - 'openrouter' (API; requires OPENROUTER_API_KEY)
# - 'huggingface' (local transformers model)

TRANSLATION_PROVIDER = "openrouter"  # or "huggingface"
TRANSLATION_MODEL = "google/gemini-3-flash-preview"
TRANSLATION_API_KEY = os.getenv("OPENROUTER_API_KEY")
TRANSLATION_MAX_WORKERS = 10

# Path to fasttext language ID model:
# https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
FASTTEXT_MODEL = paths["models"] / "lid.176.bin"


### 2.2 Language Detection

Detect language tags for configured text columns.
Only non-target-language rows are translated in the next step.


In [ ]:
if START_AT_PHASE <= 2:
    from ads_bib.translate import detect_languages
    
    print("=== Publications ===")
    publications = detect_languages(publications, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)
    
    print("\n=== References ===")
    refs = detect_languages(refs, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)
else:
    print(f"Skipping Phase 2 Translation (START_AT_PHASE={START_AT_PHASE})")

### 2.3 Translate Non-English Entries

Translate non-English rows and track token/cost metadata.
This harmonizes text fields for downstream NLP and topic modeling.


In [ ]:
if START_AT_PHASE <= 2:
    if TRANSLATION_PROVIDER not in {"openrouter", "huggingface"}:
        raise ValueError("TRANSLATION_PROVIDER must be 'openrouter' or 'huggingface'.")
    if TRANSLATION_PROVIDER == "openrouter" and not TRANSLATION_API_KEY:
        raise ValueError("OPENROUTER_API_KEY is required when TRANSLATION_PROVIDER='openrouter'.")
    if TRANSLATION_PROVIDER == "huggingface":
        try:
            import transformers  # noqa: F401
        except Exception as exc:
            raise ImportError(
                "TRANSLATION_PROVIDER='huggingface' requires 'transformers', 'torch', and 'accelerate'."
            ) from exc

    from ads_bib.translate import translate_dataframe
    from ads_bib._utils.checkpoints import save_translated_checkpoint

    print("=== Translating Publications ===")
    publications, cost_pubs = translate_dataframe(
        publications, ["Title", "Abstract"],
        provider=TRANSLATION_PROVIDER,
        model=TRANSLATION_MODEL,
        api_key=TRANSLATION_API_KEY,
        max_workers=TRANSLATION_MAX_WORKERS,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )

    print("\n=== Translating References ===")
    refs, cost_refs = translate_dataframe(
        refs, ["Title", "Abstract"],
        provider=TRANSLATION_PROVIDER,
        model=TRANSLATION_MODEL,
        api_key=TRANSLATION_API_KEY,
        max_workers=TRANSLATION_MAX_WORKERS,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )

    pub_cost = cost_pubs.get("cost_usd")
    ref_cost = cost_refs.get("cost_usd")
    print("\nTranslation Cost Snapshot:")
    print(f"  Publications: {'n/a' if pub_cost is None else f'${pub_cost:.4f}'}")
    print(f"  References:   {'n/a' if ref_cost is None else f'${ref_cost:.4f}'}")

    save_translated_checkpoint(
        publications,
        refs,
        cache_dir=paths["cache"],
        run_data_dir=run.paths["data"],
    )

else:
    print(f"Skipping Phase 2 Translation (START_AT_PHASE={START_AT_PHASE})")


---
# Phase 3: Tokenization

Create `full_text` (Title + Abstract) and tokenize publications with spaCy.
References are intentionally skipped in this phase for runtime stability.

In [ ]:
import os

TOKENIZATION_SPACY_MODEL = "en_core_web_md"
TOKENIZATION_BATCH_SIZE = 512
TOKENIZATION_N_PROCESS = min(max((os.cpu_count() or 1) - 1, 1), 8)
TOKENIZATION_DISABLE = ("ner", "parser", "textcat")

if START_AT_PHASE <= 3:
    from ads_bib._utils.checkpoints import load_translated_checkpoint
    from ads_bib.tokenize import ensure_spacy_model, tokenize_texts

    if START_AT_PHASE == 3:
        print("Loading translated data from global cache (START_AT_PHASE=3) ...")
        publications, refs = load_translated_checkpoint(
            cache_dir=paths["cache"],
            run_data_dir=run.paths["data"],
        )

    model_to_use, preloaded_nlp = ensure_spacy_model(
        spacy_model=TOKENIZATION_SPACY_MODEL,
        fallback_model="en_core_web_lg",
        spacy_disable=TOKENIZATION_DISABLE,
        auto_download=True,
    )

    print("Tokenizing publications only ...")
    publications = tokenize_texts(
        publications,
        spacy_model=model_to_use,
        nlp=preloaded_nlp,
        batch_size=TOKENIZATION_BATCH_SIZE,
        n_process=TOKENIZATION_N_PROCESS,
        spacy_disable=TOKENIZATION_DISABLE,
    )
    print("refs tokenization skipped by design.")
else:
    print(f"Skipping Phase 3 Tokenization (START_AT_PHASE={START_AT_PHASE})")


### Checkpoint: Save Phase 3 Results

Persist tokenized publications and translated references for Phase 4 restarts.
This avoids rerunning API-heavy earlier phases.


In [ ]:
if START_AT_PHASE <= 3:
    from ads_bib._utils.checkpoints import save_phase3_checkpoint

    save_phase3_checkpoint(
        publications,
        refs,
        cache_dir=paths["cache"],
        run_data_dir=run.paths["data"],
    )


---
# Phase 4: Author Name Disambiguation (Placeholder)

This step will use an external AND package once it's ready.
For now, the pipeline continues without disambiguation.

In [ ]:
# === AND PLACEHOLDER ===
# Uncomment when the AND package is available:
#
# from and_package import disambiguate
# publications = disambiguate(publications)
# refs = disambiguate(refs)

print("AND step skipped (placeholder). Continuing without disambiguation.")

---
# Phase 5: Topic Modeling & Curation

Use BERTopic to discover thematic clusters, visualize with datamapplot,
then remove unwanted clusters to curate the dataset.

**Important:** Set parameters below based on your dataset size from Phase 1.

### 5.1 Embedding Configuration

Configure embedding provider/model and optional sampling.
These settings strongly affect topic quality and runtime/cost.


In [3]:
# --- EMBEDDING CONFIGURATION ---
# Providers:
# - 'openrouter' (API; requires OPENROUTER_API_KEY)
# - 'huggingface_api' (remote API via LiteLLM)
# - 'local' (SentenceTransformers on local hardware)

EMBEDDING_PROVIDER = "openrouter"
EMBEDDING_MODEL = "google/gemini-embedding-001"
EMBEDDING_API_KEY = os.getenv("OPENROUTER_API_KEY")
EMBEDDING_MAX_WORKERS = 10  # Used for provider='openrouter'.

# For testing: set SAMPLE_SIZE to an integer (e.g. 200). None = full dataset.
SAMPLE_SIZE = None

### 5.2 Compute Embeddings

Create or load cached embeddings for `full_text`.
Caching keeps repeated runs fast and reproducible.


In [4]:
from ads_bib._utils.io import load_json_lines
import shutil

if START_AT_PHASE >= 4:
    # We skipped the earlier phases, so we must load the data from global cache
    global_pub = paths["cache"] / "publications_translated_tokenized.json"
    global_ref = paths["cache"] / "references_translated_tokenized.json"
    
    if not global_pub.exists():
        raise FileNotFoundError(f"Cannot skip to Phase 4. Global cache missing: {global_pub}")
        
    print("Loading data from global cache (Phase 1-3 skipped) ...")
    publications = load_json_lines(global_pub)
    refs = load_json_lines(global_ref)
    
    # Snapshot into current run for provenance
    shutil.copy(global_pub, run.paths["data"] / "publications_translated_tokenized.json")
    shutil.copy(global_ref, run.paths["data"] / "references_translated_tokenized.json")

print(f"Loaded {len(publications):,} publications")
print(f"Loaded {len(refs):,} references")


Loading data from global cache (Phase 1-3 skipped) ...
Loaded 87,095 publications
Loaded 307,878 references


In [5]:
from ads_bib.topic_model import compute_embeddings

df = publications.copy()
if SAMPLE_SIZE is not None:
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"SAMPLING: {len(df):,} documents")

if EMBEDDING_PROVIDER not in {"local", "huggingface_api", "openrouter"}:
    raise ValueError("EMBEDDING_PROVIDER must be 'local', 'huggingface_api', or 'openrouter'.")
if EMBEDDING_PROVIDER == "openrouter" and not EMBEDDING_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is required when EMBEDDING_PROVIDER='openrouter'.")
if EMBEDDING_PROVIDER == "local":
    try:
        import sentence_transformers  # noqa: F401
    except Exception as exc:
        raise ImportError("EMBEDDING_PROVIDER='local' requires 'sentence-transformers'.") from exc
if EMBEDDING_PROVIDER in {"openrouter", "huggingface_api"}:
    try:
        import litellm  # noqa: F401
    except Exception as exc:
        raise ImportError("Remote embedding providers require 'litellm'.") from exc

documents = df["full_text"].tolist()

embeddings = compute_embeddings(
    documents,
    provider=EMBEDDING_PROVIDER,
    model=EMBEDDING_MODEL,
    cache_dir=paths["embeddings_cache"],
    max_workers=EMBEDDING_MAX_WORKERS,
    api_key=EMBEDDING_API_KEY,
    openrouter_cost_mode=OPENROUTER_COST_MODE,
    cost_tracker=tracker,
)
print(f"Embeddings shape: {embeddings.shape}")

  Loaded embeddings from cache: embeddings_openrouter_google_gemini-embedding-001.npz
Embeddings shape: (87095, 3072)


### 5.3 Dimensionality Reduction Configuration

- Methods: `pacmap` (fast) or `umap` (preferred for hierarchical Toponymy structure).
- Heuristic: smaller/sparser sets use `n_neighbors` ~15; larger/denser sets often benefit from 50-60.
- Use `min_dist=0.0` for 5D clustering vectors and `min_dist=0.1` for 2D visualization.


In [8]:
# --- DIMENSIONALITY REDUCTION ---
DIM_REDUCTION_METHOD = "pacmap"  # PaCMAP als lokaler/globaler Kompromiss für großes GRG-Korpus

# PaCMAP nutzt 'distance' statt UMAP-Parametern wie 'min_dist' oder 'metric'.
PARAMS_5D = dict(n_neighbors=60, distance="angular", random_state=RANDOM_SEED)
PARAMS_2D = dict(n_neighbors=60, distance="angular", random_state=RANDOM_SEED)

model_safe = EMBEDDING_MODEL.replace('/', '_')
CACHE_SUFFIX = f"{EMBEDDING_PROVIDER}_{model_safe}_{DIM_REDUCTION_METHOD}"


In [9]:
from ads_bib.topic_model import reduce_dimensions

reduced_5d, reduced_2d = reduce_dimensions(
    embeddings,
    method=DIM_REDUCTION_METHOD,
    params_5d=PARAMS_5D,
    params_2d=PARAMS_2D,
    random_state=RANDOM_SEED,
    cache_dir=paths["dim_reduction_cache"],
    cache_suffix=CACHE_SUFFIX,
)

Note: `n_components != 2` have not been thoroughly tested.


  Computing 5d_openrouter_google_gemini-embedding-001_pacmap with PACMAP ...
Applied PCA, the dimensionality becomes 100
PaCMAP(n_neighbors=60, n_MN=30, n_FP=60, distance=angular, lr=1.0, n_iters=(100, 100, 250), apply_pca=True, opt_method='adam', verbose=True, intermediate=False, seed=42)
Finding pairs
Found nearest neighbor
Calculated sigma
Found scaled dist
Pairs sampled successfully.
((5225700, 2), (2612850, 2), (5225700, 2))
Initial Loss: 3842549.0
Iteration:   10, Loss: 3319797.250000
Iteration:   20, Loss: 2931242.500000
Iteration:   30, Loss: 2767678.750000
Iteration:   40, Loss: 2652001.500000
Iteration:   50, Loss: 2542048.250000
Iteration:   60, Loss: 2428601.500000
Iteration:   70, Loss: 2304294.000000
Iteration:   80, Loss: 2160532.750000
Iteration:   90, Loss: 1982858.625000
Iteration:  100, Loss: 1732805.625000
Iteration:  110, Loss: 2332553.000000
Iteration:  120, Loss: 2283313.000000
Iteration:  130, Loss: 2265842.500000
Iteration:  140, Loss: 2260343.250000
Iteration:

Iteration:  450, Loss: 1001194.500000
Elapsed time: 190.20s
  Saved: reduced_5d_openrouter_google_gemini-embedding-001_pacmap.npy
  Computing 2d_openrouter_google_gemini-embedding-001_pacmap with PACMAP ...
Applied PCA, the dimensionality becomes 100
PaCMAP(n_neighbors=60, n_MN=30, n_FP=60, distance=angular, lr=1.0, n_iters=(100, 100, 250), apply_pca=True, opt_method='adam', verbose=True, intermediate=False, seed=42)
Finding pairs
Found nearest neighbor
Calculated sigma
Found scaled dist
Pairs sampled successfully.
((5225700, 2), (2612850, 2), (5225700, 2))
Initial Loss: 3842549.0
Iteration:   10, Loss: 3175957.750000
Iteration:   20, Loss: 2932044.750000
Iteration:   30, Loss: 2802943.500000
Iteration:   40, Loss: 2701685.750000
Iteration:   50, Loss: 2601836.500000
Iteration:   60, Loss: 2493773.500000
Iteration:   70, Loss: 2371132.500000
Iteration:   80, Loss: 2226970.750000
Iteration:   90, Loss: 2042297.750000
Iteration:  100, Loss: 1769453.000000
Iteration:  110, Loss: 2301357.0

### 5.4 Clustering Configuration

**Adjust clustering granularity based on dataset size.**
- `MIN_CLUSTER_SIZE` controls BERTopic/HDBSCAN topic granularity (roughly ~0.1% of docs).
- `BASE_MIN_CLUSTER_SIZE` controls Toponymy micro-cluster granularity.
- `TOPONYMY_LAYER_INDEX` selects the fallback primary layer for visualization.


In [10]:
# --- CLUSTERING CONFIGURATION ---
CLUSTERING_METHOD = "fast_hdbscan"  # Schnellste Variante für 87k

n_docs = len(df)
MIN_CLUSTER_SIZE = max(15, int(n_docs * 0.001)) # ca. 87 bei 87k Dokumenten
BASE_MIN_CLUSTER_SIZE = max(55, int(n_docs * 0.0007))
print(f"Dataset: {n_docs:,} documents")
print(f"  MIN_CLUSTER_SIZE={MIN_CLUSTER_SIZE}")
print(f"  BASE_MIN_CLUSTER_SIZE={BASE_MIN_CLUSTER_SIZE}")

CLUSTER_PARAMS = dict(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=2, # Reduziert Outlier drastisch, da weniger Dichte für Cluster-Zugehörigkeit gefordert wird
    cluster_selection_method="eom",
    cluster_selection_epsilon=0.05, # Absorbiert Randpunkte in bestehende Cluster
)

TOPONYMY_CLUSTER_PARAMS = dict(
    min_clusters=10,
    min_samples=2,
    base_min_cluster_size=BASE_MIN_CLUSTER_SIZE,
)

# EVoC uses a backend-specific constructor in some versions.
# Keep this dict conservative; unsupported keys are filtered in fit_toponymy.
TOPONYMY_EVOC_CLUSTER_PARAMS = dict(
    min_clusters=10,
    min_samples=2,
    base_min_cluster_size=BASE_MIN_CLUSTER_SIZE,
    noise_level=0.35,
    n_neighbors=15,
    n_epochs=35,
)

TOPONYMY_LAYER_INDEX = 1  # Broader Toponymy layer to reduce -1 outliers on large corpora.


Dataset: 87,095 documents
  MIN_CLUSTER_SIZE=87
  BASE_MIN_CLUSTER_SIZE=60


### 5.5 Backend & LLM Configuration

Backend matrix:
- `bertopic`: BERTopic on 5D reduced vectors + optional outlier reassignment refresh.
- `toponymy`: Toponymy + `ToponymyClusterer` on 5D reduced vectors.
- `toponymy_evoc`: Toponymy + `EVoCClusterer` on raw embeddings (no 5D clustering vectors).

LLM provider matrix:
- BERTopic: `local`, `huggingface_api`, `openrouter`
- Toponymy / Toponymy+EVoC: `local` (Toponymy HuggingFaceNamer) or `openrouter`

`MIN_DF` guidance:
- small sample runs: often `1`
- full runs: usually `2+`


In [11]:
# --- TOPIC BACKEND + LLM LABELING ---
TOPIC_BACKEND = "bertopic"  # Explizit BERTopic wie gewünscht

# BERTopic providers: 'local', 'huggingface_api', 'openrouter'
# Toponymy providers: 'local' (HuggingFaceNamer) or 'openrouter'
LLM_PROVIDER = "openrouter"
LLM_MODEL = "google/gemini-3-flash-preview"
LLM_API_KEY = os.getenv("OPENROUTER_API_KEY")

PIPELINE_MODELS = ["POS", "KeyBERT", "MMR"]
PARALLEL_MODELS = ["MMR", "POS", "KeyBERT"]

TOPONYMY_EMBEDDING_MODEL = EMBEDDING_MODEL
TOPONYMY_MAX_WORKERS = 10  # Shared OpenRouter worker count for Toponymy labeling + embeddings.

# MIN_DF=5 filtert extrem seltene Wörter heraus. Beschleunigt c-TF-IDF bei 87k enorm und spart RAM.
MIN_DF = 5


In [12]:
from ads_bib.topic_model import fit_bertopic, fit_toponymy, reduce_outliers, build_topic_dataframe
import numpy as np

valid_backends = {"bertopic", "toponymy", "toponymy_evoc"}
if TOPIC_BACKEND not in valid_backends:
    raise ValueError(
        f"Invalid TOPIC_BACKEND '{TOPIC_BACKEND}'. Use 'bertopic', 'toponymy', or 'toponymy_evoc'."
    )

allowed_llm = (
    {"local", "huggingface_api", "openrouter"}
    if TOPIC_BACKEND == "bertopic"
    else {"local", "openrouter"}
)
if LLM_PROVIDER not in allowed_llm:
    raise ValueError(
        f"LLM_PROVIDER '{LLM_PROVIDER}' is not valid for TOPIC_BACKEND='{TOPIC_BACKEND}'. "
        f"Allowed: {sorted(allowed_llm)}"
    )
if LLM_PROVIDER == "openrouter" and not LLM_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is required when LLM_PROVIDER='openrouter'.")

if LLM_PROVIDER == "local":
    if TOPIC_BACKEND == "bertopic":
        try:
            import transformers  # noqa: F401
        except Exception as exc:
            raise ImportError("BERTopic local labeling requires 'transformers'.") from exc
    else:
        try:
            import sentence_transformers  # noqa: F401
            from toponymy.llm_wrappers import HuggingFaceNamer  # noqa: F401
        except Exception as exc:
            raise ImportError(
                "Toponymy local labeling requires 'sentence-transformers' and Toponymy HuggingFaceNamer support."
            ) from exc

selected_toponymy_cluster_params = (
    TOPONYMY_EVOC_CLUSTER_PARAMS
    if TOPIC_BACKEND == "toponymy_evoc"
    else TOPONYMY_CLUSTER_PARAMS
)

if TOPIC_BACKEND == "bertopic":
    topic_model = fit_bertopic(
        documents,
        reduced_5d,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        pipeline_models=PIPELINE_MODELS,
        parallel_models=PARALLEL_MODELS,
        min_df=MIN_DF,
        clustering_method=CLUSTERING_METHOD,
        clustering_params=CLUSTER_PARAMS,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )

    topics = np.array(topic_model.topics_)
    topics = reduce_outliers(
        topic_model,
        documents,
        topics,
        reduced_5d,
        threshold=0.4, # Aggressivere Outlier-Zuweisung (vorher 0.8). Zwingt mehr Dokumente in Cluster.
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )
    topic_info = topic_model.get_topic_info()

elif TOPIC_BACKEND in {"toponymy", "toponymy_evoc"}:
    topic_model, topics, topic_info = fit_toponymy(
        documents,
        embeddings,
        reduced_5d,
        backend=TOPIC_BACKEND,
        layer_index=TOPONYMY_LAYER_INDEX,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        embedding_model=TOPONYMY_EMBEDDING_MODEL,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        max_workers=TOPONYMY_MAX_WORKERS,
        clusterer_params=selected_toponymy_cluster_params,
        cost_tracker=tracker,
    )

else:
    raise ValueError(
        f"Invalid TOPIC_BACKEND '{TOPIC_BACKEND}'. "
        "Use 'bertopic', 'toponymy', or 'toponymy_evoc'."
    )

df = build_topic_dataframe(
    df,
    topic_model,
    topics,
    reduced_2d,
    embeddings,
    topic_info=topic_info,
)
topic_info

Preparing BERTopic components (pipeline=['POS', 'KeyBERT', 'MMR'], parallel=['MMR', 'POS', 'KeyBERT']) ...
  Initializing POS keyword extraction model: en_core_web_sm


2026-02-23 13:43:44,113 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-23 13:43:44,114 - BERTopic - Dimensionality - Completed ✓
2026-02-23 13:43:44,116 - BERTopic - Cluster - Start clustering the reduced embeddings


Fitting BERTopic (LLM: openrouter/google/gemini-3-flash-preview) ...


2026-02-23 13:43:50,904 - BERTopic - Cluster - Completed ✓
2026-02-23 13:43:50,923 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 101/101 [03:31<00:00,  2.09s/it]
2026-02-23 13:53:26,422 - BERTopic - Representation - Completed ✓


  OpenRouter cost: $0.0274 (101/101 priced; direct=101, fetched=0, mode=hybrid)
  Outliers: 47,203 → 0
  Refreshing topic representations after outlier reassignment (not a full BERTopic refit).
  Topic reduction must happen before this step when using manual topic assignments.


100%|██████████| 100/100 [03:04<00:00,  1.84s/it]


  OpenRouter cost: $0.0269 (100/100 priced; direct=100, fetched=0, mode=hybrid)


,Topic,Count,Name,Representation,MMR,POS,KeyBERT,Representative_Docs
0,0,3588,0_Alternative and Mathematical Formulations of...,[Alternative and Mathematical Formulations of ...,"[tensor, general relativity, gravitational fie...","[theory, relativity, field, equations, general...","[general relativity, gravitational field, theo...",[Aoverline {Poincaracute{e}} gauge theory of g...
1,1,2114,1_Geodetic Satellite Missions and Gravity Mode...,[Geodetic Satellite Missions and Gravity Model...,"[geodetic, satellite, gravity field, accuracy,...","[gravity, geodetic, satellite, earth, data, ac...","[gravity data, earth gravity field, gravity me...",[GRIM4: A model of the Earth's gravity field.....
2,2,2160,2_Crustal Structure and Lithospheric Tectonic ...,[Crustal Structure and Lithospheric Tectonic E...,"[crust, seismic, mantle, rift, zone, ridge, li...","[crust, km, crustal, basin, seismic, mantle, r...","[tectonic, geological, upper mantle, geophysic...",[A tomographic study of the lithosphere beneat...
3,3,1819,3_Fluid Dynamics and Heat Transfer in Microgra...,[Fluid Dynamics and Heat Transfer in Micrograv...,"[liquid, convection, microgravity, drop, heat ...","[liquid, flow, fluid, surface, convection, hea...","[convection, fluids, viscous, heat transfer, b...",[Thermocapillary flow and gaseous convection i...
4,4,2251,4_Quantum Mechanics and Thermodynamics of Blac...,[Quantum Mechanics and Thermodynamics of Black...,"[black hole, black holes, quantum, hawking, sp...","[black, black hole, hole, holes, black holes, ...","[black hole entropy, black holes, dimensional ...",[Black Hole Entropy and Physics at Planckian S...
...,...,...,...,...,...,...,...,...
95,95,308,95_Perturbations and Mathematical Theory of Bl...,[Perturbations and Mathematical Theory of Blac...,"[black hole, black holes, kerr black, kerr bla...","[black, hole, black hole, perturbations, gravi...","[kerr black hole, schwarzschild black hole, ro...",[Gravitational radiation from an extreme Kerr ...
96,96,167,96_GRAPE Special-Purpose Computers for N-body,[GRAPE Special-Purpose Computers for N-body],"[grape, tree, algorithm, parallel, body simula...","[body, simulations, code, tree, algorithm, par...","[body simulations, simulations, grape, body si...",[GRAPE: special purpose computer for simulatio...
97,97,338,97_Computational Mechanics and Structural Mate...,[Computational Mechanics and Structural Materi...,"[materials, stress, crack, finite element, bea...","[materials, stress, crack, strain, unified, fi...","[elastic, stresses, deformation, fracture, fin...",[Development of methods for predicting large c...
98,98,561,98_Cosmic Microwave Background Anisotropies an...,[Cosmic Microwave Background Anisotropies and ...,"[microwave, microwave background, cosmic, cosm...","[universe, microwave, background, cosmic, cosm...","[cosmic microwave background, cosmology, cosmo...",[Large-scale polarization of the cosmic microw...


### 5.6 Visualize Topics

Render an interactive topic map from 2D coordinates and topic labels.
Use this view to inspect cluster semantics before curation.


In [13]:
from ads_bib.visualize import create_topic_map

# Use all layers for Toponymy, or just 'Name' for BERTopic
if TOPIC_BACKEND in {"toponymy", "toponymy_evoc"}:
    num_layers = len([col for col in df.columns if col.startswith("Topic_Layer_")])
    label_column = [f"Topic_Layer_{i}" for i in range(num_layers)]
else:
    label_column = "Name"

plot = create_topic_map(
    df,
    label_column=label_column,
    title="ADS Bibliometric Map",
    subtitle=f"Topics labeled with {LLM_PROVIDER}/{LLM_MODEL}",
    dark_mode=True,
    output_path=run.paths["plots"] / "topic_map.html",
)
# plot  # uncomment to display inline

Saved: topic_map.html


### 5.7 Curate Dataset

Review the cluster summary, then remove clusters that don't fit your research question.

In [14]:
from ads_bib.curate import get_cluster_summary, remove_clusters

get_cluster_summary(df)

,topic_id,Count,Percentage,Label
0,0,3588,4.1,0_Alternative and Mathematical Formulations of...
54,54,2280,2.6,54_Mathematical Analysis of General Relativity...
4,4,2251,2.6,4_Quantum Mechanics and Thermodynamics of Blac...
15,15,2228,2.6,15_Exact Solutions for Bianchi Cosmological Mo...
12,12,2182,2.5,12_Exact Solutions of Einstein-Maxwell Field E...
...,...,...,...,...
92,92,266,0.3,92_Gravity Probe B Relativity Gyroscope Experi...
62,62,229,0.3,62_Space Telescopes and Astronomical Research ...
72,72,220,0.3,72_Interdisciplinary Computational Science and...
76,76,217,0.2,76_Physics Literature Education and Historical...


### Checkpoint: Save Curated Dataset

Save the curated topic dataset as the handoff for citation analysis.
This provides a stable artifact for Phase 6 and external reuse.


In [15]:
from ads_bib._utils.io import save_parquet

# Ensure References is list type for Parquet
df["References"] = df["References"].apply(lambda x: x if isinstance(x, list) else [])

save_parquet(df, run.paths["data"] / "curated_dataset.parquet")
print(f"Curated dataset: {len(df):,} documents")

Curated dataset: 87,095 documents


In [16]:
from ads_bib._utils.io import load_parquet

df = load_parquet(run.paths["data"] / "curated_dataset.parquet")
df.head(10)

,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,...,full_text,tokens,embedding_2d_x,embedding_2d_y,topic_id,Name,MMR,POS,KeyBERT,full_embeddings
0,2000A&A...363.1081C,"[Claret, A.]",A new non-linear limb-darkening law for LTE st...,2000,Astronomy and Astrophysics,A&A,,363,1081,1190,...,A new non-linear limb-darkening law for LTE st...,"[new, non, linear, limb, darken, law, lte, ste...",2.117913,-7.369764,7,7_Spectroscopic Analysis of Stellar Evolution ...,"stars, stellar, spectra, white dwarf, atmosphe...","stars, star, white, stellar, dwarf, lines, abu...","white dwarf, stellar, white dwarfs, stars, spe...","[0.011602388, 0.0061915503, 0.0024698696, -0.0..."
1,2000PhRvL..85.5042P,"[Parikh, Maulik K., Wilczek, Frank]",Hawking Radiation As Tunneling,2000,Physical Review Letters,PhRvL,24,85,5042,5045,...,Hawking Radiation As Tunneling. We present a s...,"[hawk, radiation, tunneling, present, short, d...",-4.954878,-0.708783,4,4_Quantum Mechanics and Thermodynamics of Blac...,"black hole, black holes, quantum, hawking, spa...","black, black hole, hole, holes, black holes, e...","black hole entropy, black holes, dimensional b...","[-0.004152307, 0.006941758, 0.027285794, -0.09..."
2,2000PhRvL..85.4438A,"[ArmendarizPicon, C., Mukhanov, V., Steinhardt...",Dynamical Solution to the Problem of a Small C...,2000,Physical Review Letters,PhRvL,21,85,4438,4441,...,Dynamical Solution to the Problem of a Small C...,"[dynamical, solution, problem, small, cosmolog...",-4.499674,-3.867481,13,13_Models and Dynamics of Inflationary Cosmology,"inflation, inflationary, cosmological, scalar ...","inflation, inflationary, universe, scalar, cos...","inflationary cosmology, inflationary universe,...","[0.008425553, 0.0017861549, -0.002557501, -0.0..."
3,2000ApJ...542..281A,"[Alcock, C., Allsman, R. A., Alves, D. R., Axe...",The MACHO Project: Microlensing Results from 5...,2000,The Astrophysical Journal,ApJ,1,542,281,307,...,The MACHO Project: Microlensing Results from 5...,"[macho, project, microlensing, result, year, l...",-0.512256,-9.881772,46,46_Gravitational Microlensing and Galactic Dar...,"stars, microlensing events, halo, gravitationa...","microlensing, events, bulge, stars, microlensi...","gravitational microlensing, galactic halo, mic...","[-0.030304352, 0.004380863, -0.0031965098, -0...."
4,2000PhRvD..62h4003V,"[Virbhadra, K. S., Ellis, George F. R.]",Schwarzschild black hole lensing,2000,Physical Review D,PhRvD,8,62,084003,,...,Schwarzschild black hole lensing. We study str...,"[schwarzschild, black, hole, lense, study, str...",-1.176879,-8.457943,27,27_Unified Models of Active Galactic Nuclei,"emission, quasars, galaxies, galactic, luminos...","radio, ray, emission, sources, jet, quasars, g...","radio galaxies, galactic nuclei, active galact...","[-0.0005508098, -0.016840542, 0.003519197, -0...."
5,2000PhLB..493..149A,"[AyónBeato, E., García, A.]",The Bardeen model as a nonlinear magnetic mono...,2000,Physics Letters B,PhLB,1,493,149,152,...,The Bardeen model as a nonlinear magnetic mono...,"[bardeen, model, nonlinear, magnetic, monopole...",-2.800488,-1.315550,99,99_Einstein-Yang-Mills and Dilaton Black Hole,"solutions, black holes, dilaton, einstein yang...","solutions, black, black hole, hole, black hole...","black hole solutions, einstein yang mills, sol...","[-0.002997447, 0.0022824937, 0.014165188, -0.0..."
6,2000PhRvD..62f4015B,"[Buonanno, Alessandra, Damour, Thibault]",Transition from inspiral to plunge in binary b...,2000,Physical Review D,PhRvD,6,62,064015,,...,Transition from inspiral to plunge in binary b...,"[transition, inspiral, plunge, binary, black, ...",-0.053508,-2.760789,37,37_Numerical Relativity and Black Hole Dynamics,"black hole, numerical relativity, black holes,...","black, numerical, hole, black hole, holes, num...","black hole spacetimes, black holes, binary bla...","[-0.005866812, 0.0062490753, 0.004115057, -0.0..."
7,2000PhRvL..85.2236B,"[Boisseau, B., EspositoFarèse

---
# Phase 6: Citation Networks

Compute citation networks from the curated dataset and export
to GEXF (Gephi), Graphology JSON (Sigma.js), CSV, and/or Web of Science format.

### 6.1 Citation Configuration

Set network metrics, thresholds, filters, and export formats.
These parameters define which citation structures are constructed.


In [17]:
# --- CITATION CONFIGURATION ---
CITE_METRICS = ["direct", "co_citation", "bibliographic_coupling", "author_co_citation"]
MIN_COUNTS = {
    "direct": 1,
    "co_citation": 1,
    "bibliographic_coupling": 1,
    "author_co_citation": 1,
}
AUTHORS_FILTER = None
OUTPUT_FORMAT = "gexf"  # 'gexf', 'graphology', 'csv', 'all'
CLUSTERS_TO_REMOVE = []

# Snapshot the full configuration once all phase configs are defined.
run.save_config(locals())


Snapshot of configuration saved to config_used.yaml (34 parameters tracked).


### 6.2 Build Node Table & Compute Networks

Build node metadata and compute selected citation networks.
Outputs are exported for Gephi/Sigma/CSV workflows.


In [18]:
from ads_bib.citations import build_all_nodes, process_all_citations, export_wos_format

# Reload processed data if needed
all_nodes = build_all_nodes(publications, refs)

results = process_all_citations(
    bibcodes, references, publications, refs, all_nodes,
    metrics=CITE_METRICS,
    min_counts=MIN_COUNTS,
    authors_filter=AUTHORS_FILTER,
    output_format=OUTPUT_FORMAT,
    output_dir=run.paths["data"],
)

All nodes: 355,989


NameError: name 'bibcodes' is not defined

### 6.3 Export Web of Science Format

Export the curated corpus in WOS text format.
This supports downstream tooling such as CiteSpace/VOSviewer.


In [19]:
suffix = "_filtered" if AUTHORS_FILTER else ""
export_wos_format(
    publications, refs,
    output_path=run.paths["data"] / f"download_wos_export{suffix}.txt",
)

  WOS format: download_wos_export.txt


---
# Summary

Final dataset statistics and output file listing.

In [20]:
import os

print("="*60)
print("PIPELINE COMPLETE")
print("="*60)
print(f"Publications:     {len(publications):,}")
print(f"References:       {len(refs):,}")
print(f"Curated dataset:  {len(df):,}")
print(f"Topics found:     {df['topic_id'].nunique()}")
print()
print("Output files:")
for root, dirs, files in os.walk(run.paths["root"]):
    for f in sorted(files):
        fpath = Path(root) / f
        size_mb = fpath.stat().st_size / 1024 / 1024
        print(f"  {fpath.relative_to(run.paths['root'])} ({size_mb:.1f} MB)")

print()
print(tracker.compact_summary())


PIPELINE COMPLETE
Publications:     87,095
References:       307,878
Curated dataset:  87,095
Topics found:     100

Output files:
  config_used.yaml (0.0 MB)
  data\curated_dataset.parquet (1197.6 MB)
  data\download_wos_export.txt (174.4 MB)
  data\publications_translated_tokenized.json (398.6 MB)
  data\references_translated_tokenized.json (649.8 MB)
  plots\topic_map.html (38.4 MB)

CostTracker Summary (compact)
  llm_labeling | google/gemini-3-flash-preview | tokens(total=49,601, prompt=48,578, completion=1,023) | calls=1 | cost=$0.0274
  llm_labeling_post_outliers | google/gemini-3-flash-preview | tokens(total=48,817, prompt=47,816, completion=1,001) | calls=1 | cost=$0.0269
  TOTAL | all models | tokens(total=98,418, prompt=-, completion=-) | calls=2 | cost=$0.0543
